# 17 — AgentCore Telemetry & Observability

Walk through the full telemetry picture for OrgIQ agents:
- CloudWatch Logs: structured traces from AgentCore runtimes
- X-Ray: distributed traces across Lambda tools + AgentCore calls
- Online Evaluation: live quality metrics via `CfnOnlineEvaluationConfig`
- Custom evaluators: LLM-as-judge scoring for org-specific quality

> **Prerequisites:** Deploy `OrgIQ-AgentCoreTelemetry` stack first:
> ```bash
> cd infra && npx cdk deploy OrgIQ-AgentCoreTelemetry
> ```
> Enable online evaluation after deploying runtimes:
> ```bash
> npx cdk deploy OrgIQ-AgentCoreTelemetry --context enableEval=true
> ```

## Step 1: Load Config & Clients

In [ ]:
import boto3
import json
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

config = json.loads(Path('config.json').read_text())
REGION = config['REGION']
PREFIX = config['PROJECT_PREFIX']
ACCOUNT_ID = config['ACCOUNT_ID']

logs_client = boto3.client('logs', region_name=REGION)
xray_client = boto3.client('xray', region_name=REGION)
bedrock_agentcore = boto3.client('bedrock-agentcore', region_name=REGION)
cw_client = boto3.client('cloudwatch', region_name=REGION)

AGENT_NAMES = ['supervisor', 'search', 'scoring', 'advisor', 'ingestion', 'graph-builder']
SAFE_NAMES = [n.replace('-', '_') for n in AGENT_NAMES]

print(f'✓ Clients ready | region={REGION} | prefix={PREFIX}')

## Step 2: List Log Groups & Recent Activity

In [ ]:
# Discover all AgentCore log groups for this project
paginator = logs_client.get_paginator('describe_log_groups')
log_groups = []
for page in paginator.paginate(logGroupNamePrefix=f'/aws/bedrock-agentcore/runtimes/{PREFIX}_'):
    log_groups.extend(page['logGroups'])

print(f'Found {len(log_groups)} AgentCore log groups:')
for lg in log_groups:
    size_mb = lg.get('storedBytes', 0) / (1024 * 1024)
    print(f"  {lg['logGroupName']}  ({size_mb:.2f} MB stored)")

if not log_groups:
    print('  (none yet — deploy runtimes and invoke agents first)')

## Step 3: Query Recent Agent Traces from CloudWatch Logs

In [ ]:
# CloudWatch Logs Insights query — find recent invocations across all agents
query = """
fields @timestamp, agentName, sessionId, inputTokens, outputTokens, latencyMs, @message
| filter @message like /INVOCATION_COMPLETE/
| sort @timestamp desc
| limit 20
"""

log_group_names = [f'/aws/bedrock-agentcore/runtimes/{PREFIX}_{n}' for n in SAFE_NAMES]
# Filter to existing groups only
existing = {lg['logGroupName'] for lg in log_groups}
log_group_names = [g for g in log_group_names if g in existing]

if log_group_names:
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=24)

    resp = logs_client.start_query(
        logGroupNames=log_group_names,
        startTime=int(start_time.timestamp()),
        endTime=int(end_time.timestamp()),
        queryString=query,
    )
    query_id = resp['queryId']

    print(f'Query started: {query_id}')
    for _ in range(10):
        time.sleep(2)
        result = logs_client.get_query_results(queryId=query_id)
        if result['status'] in ('Complete', 'Failed', 'Cancelled'):
            break

    if result['status'] == 'Complete':
        rows = result['results']
        print(f'\n✓ {len(rows)} invocations in the last 24h:')
        for row in rows:
            fields = {f['field']: f['value'] for f in row}
            ts = fields.get('@timestamp', '?')
            agent = fields.get('agentName', '?')
            latency = fields.get('latencyMs', '?')
            tokens = fields.get('outputTokens', '?')
            print(f'  [{ts}] {agent:20s} latency={latency}ms output_tokens={tokens}')
    else:
        print(f'Query status: {result["status"]}')
else:
    print('No log groups found. Run agents first to generate traces.')

## Step 4: X-Ray Traces — End-to-End Latency Breakdown

In [ ]:
# Fetch X-Ray trace summaries for the last hour
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=1)

try:
    resp = xray_client.get_trace_summaries(
        StartTime=start_time,
        EndTime=end_time,
        TimeRangeType='Event',
        FilterExpression=f'annotation.aws:bedrock-agentcore:project = "{PREFIX}"',
    )
    summaries = resp.get('TraceSummaries', [])
    print(f'✓ Found {len(summaries)} traces in the last hour')

    if summaries:
        # Fetch detailed segments for the most recent 5 traces
        trace_ids = [s['Id'] for s in summaries[:5]]
        details = xray_client.batch_get_traces(TraceIds=trace_ids)

        for trace in details.get('Traces', []):
            total_ms = trace.get('Duration', 0) * 1000
            segment_count = len(trace.get('Segments', []))
            print(f'  Trace {trace["Id"][:16]}... | {total_ms:.0f}ms | {segment_count} segments')

            # Show per-segment breakdown
            for seg in trace.get('Segments', [])[:3]:
                doc = json.loads(seg.get('Document', '{}'))
                name = doc.get('name', '?')
                duration_ms = doc.get('end_time', 0) * 1000 - doc.get('start_time', 0) * 1000
                fault = '⚠' if doc.get('fault') else ''
                print(f'    {fault} {name:40s} {duration_ms:.0f}ms')
    else:
        print('  No traces found. Invoke agents with X-Ray tracing enabled.')

except Exception as e:
    print(f'X-Ray query failed: {type(e).__name__}: {e}')
    print('  (X-Ray may not have data yet — invoke agents first)')

## Step 5: CloudWatch Metrics — AgentCore Runtime Stats

In [ ]:
from datetime import timezone

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=6)

print('AgentCore Runtime Metrics (last 6 hours):')
print(f'{"Agent":<25} {"Invocations":>12} {"Errors":>8} {"p99 Latency (ms)":>18}')
print('-' * 70)

for safe_name in SAFE_NAMES:
    runtime_name = f'{PREFIX}_{safe_name}'
    dims = [{'Name': 'AgentRuntimeName', 'Value': runtime_name}]

    def get_metric(metric_name, stat):
        try:
            resp = cw_client.get_metric_statistics(
                Namespace='bedrock-agentcore',
                MetricName=metric_name,
                Dimensions=dims,
                StartTime=start_time,
                EndTime=end_time,
                Period=21600,  # 6 hours = one bucket
                Statistics=[stat] if stat in ('Sum', 'Average', 'Maximum') else [],
                ExtendedStatistics=[stat] if stat.startswith('p') else [],
            )
            pts = resp.get('Datapoints', [])
            if not pts:
                return 0
            return pts[0].get(stat, pts[0].get('ExtendedStatistics', {}).get(stat, 0))
        except Exception:
            return '-'

    invocations = get_metric('Invocations', 'Sum')
    errors = get_metric('Errors', 'Sum')
    latency = get_metric('InvocationLatency', 'p99')

    print(f'{safe_name:<25} {str(invocations):>12} {str(errors):>8} {str(latency):>18}')

print()
print('✓ (zeros = no invocations yet or runtime not deployed)')

## Step 6: Evaluator Status — Custom Evaluators

In [ ]:
# List custom evaluators deployed by the telemetry stack
try:
    resp = bedrock_agentcore.list_evaluators()
    evaluators = resp.get('evaluators', [])
    print(f'✓ Found {len(evaluators)} evaluator(s):')
    for ev in evaluators:
        print(f"  [{ev.get('status')}] {ev.get('name')}")
        print(f"    ID: {ev.get('evaluatorId')}")
        print(f"    Level: {ev.get('level')}")
except Exception as e:
    print(f'list_evaluators failed: {type(e).__name__}: {e}')
    print('  Deploy OrgIQ-AgentCoreTelemetry first')

## Step 7: Online Evaluation Config Status

In [ ]:
# List online evaluation configs
try:
    resp = bedrock_agentcore.list_online_evaluation_configs()
    configs = resp.get('onlineEvaluationConfigs', [])
    print(f'✓ Found {len(configs)} online evaluation config(s):')
    for cfg in configs:
        print(f"  [{cfg.get('status')}] {cfg.get('name')}")
        print(f"    Evaluators: {[e.get('evaluatorId') for e in cfg.get('evaluators', [])]}")
        print(f"    Sampling: {cfg.get('rule', {}).get('samplingConfig', {}).get('samplingPercentage')}%")
    if not configs:
        print('  None yet — redeploy with --context enableEval=true after runtimes are running')
except Exception as e:
    print(f'list_online_evaluation_configs failed: {type(e).__name__}: {e}')
    print('  Deploy OrgIQ-AgentCoreTelemetry first')

## Step 8: Invoke an Agent and Observe Telemetry

In [ ]:
# Load runtime IDs from config
RUNTIME_IDS = config.get('RUNTIME_IDS', {})
SUPERVISOR_ENDPOINT_URL = config.get('SUPERVISOR_ENDPOINT_URL', '')

if not SUPERVISOR_ENDPOINT_URL:
    print('SUPERVISOR_ENDPOINT_URL not in config.json')
    print('Run notebook 14-deploy-runtime.ipynb first to push container images')
else:
    import requests
    import boto3
    from botocore.auth import SigV4Auth
    from botocore.awsrequest import AWSRequest

    session = boto3.Session(region_name=REGION)
    credentials = session.get_credentials().get_frozen_credentials()

    payload = {
        'prompt': 'Who are the top 3 Kubernetes experts in the organization?',
        'sessionId': f'telemetry-test-{int(time.time())}'
    }

    # Sign request with SigV4
    request = AWSRequest(method='POST', url=SUPERVISOR_ENDPOINT_URL, data=json.dumps(payload))
    SigV4Auth(credentials, 'bedrock-agentcore', REGION).add_auth(request)

    print(f'Invoking supervisor agent...')
    start_ts = time.time()
    resp = requests.post(
        SUPERVISOR_ENDPOINT_URL,
        headers=dict(request.headers),
        data=json.dumps(payload),
        timeout=60
    )
    elapsed = (time.time() - start_ts) * 1000

    print(f'Status: {resp.status_code} | Latency: {elapsed:.0f}ms')
    if resp.ok:
        print(f'Response snippet: {resp.text[:300]}...')
        print()
        print('Wait ~30s then re-run Steps 3-5 to see telemetry data appear.')
    else:
        print(f'Error: {resp.text[:200]}')

## Step 9: Manual Evaluation with Custom Evaluators

In [ ]:
# Get the quality evaluator ID from config or deployed stack
QUALITY_EVALUATOR_ID = config.get('QUALITY_EVALUATOR_ID', '')

if not QUALITY_EVALUATOR_ID:
    print('QUALITY_EVALUATOR_ID not in config.json')
    print('Save it from the OrgIQ-AgentCoreTelemetry stack outputs')
else:
    # Run a sample evaluation using the custom evaluator
    test_trace = {
        'input': 'Who is the best person to ask about Kubernetes?',
        'output': 'Alice Chen is the top Kubernetes expert with a strength score of 0.9. She has authored the Kubernetes Best Practices document.',
        'context': 'Knowledge graph query returned: Alice Chen (SRE) - Kubernetes 0.9, Carol Diaz (Architect) - Kubernetes 0.7'
    }

    try:
        resp = bedrock_agentcore.evaluate(
            evaluatorId=QUALITY_EVALUATOR_ID,
            input=test_trace['input'],
            output=test_trace['output'],
            context=test_trace.get('context'),
        )
        result = resp.get('evaluationResult', {})
        print(f'Quality Evaluation Result:')
        print(f'  Score: {result.get("numericalScore", result.get("categoricalScore", "?"))}')
        print(f'  Reasoning: {result.get("reasoning", "?")}')
    except Exception as e:
        print(f'Evaluate API call failed: {type(e).__name__}: {e}')

## Step 10: Online Evaluation Results (after traffic)

In [ ]:
# List online evaluation results for the last 24 hours
EVAL_CONFIG_NAME = f'{PREFIX.replace("-", "_")}_online_eval'

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=24)

try:
    resp = bedrock_agentcore.list_online_evaluation_results(
        onlineEvaluationConfigName=EVAL_CONFIG_NAME,
        startTime=start_time.isoformat(),
        endTime=end_time.isoformat(),
    )
    results = resp.get('results', [])
    print(f'✓ {len(results)} evaluation results in the last 24h')

    if results:
        # Aggregate scores per evaluator
        from collections import defaultdict
        scores_by_evaluator = defaultdict(list)
        for r in results:
            for score in r.get('evaluatorScores', []):
                ev_id = score.get('evaluatorId', '?')
                val = score.get('numericalScore') or (1 if score.get('categoricalScore') == 'Correct' else 0)
                scores_by_evaluator[ev_id].append(val)

        print('\nAverage scores per evaluator:')
        for ev_id, scores in scores_by_evaluator.items():
            avg = sum(scores) / len(scores)
            print(f'  {ev_id[:40]}: {avg:.2f} (n={len(scores)})')
except Exception as e:
    print(f'list_online_evaluation_results failed: {type(e).__name__}: {e}')
    print('  Enable eval config and generate traffic first')

## Summary

| What | Where | How |
|------|-------|-----|
| Runtime logs | `/aws/bedrock-agentcore/runtimes/{prefix}_{agent}` | CloudWatch Logs Insights |
| Distributed traces | X-Ray | `get_trace_summaries()` with bedrock-agentcore filter |
| Runtime metrics | `bedrock-agentcore` namespace | CloudWatch dashboard `orgiq-agentcore-telemetry` |
| Online evaluation | `{prefix}_online_eval` config | Automatic sampling at 10% |
| Custom scoring | `{prefix}_response_quality` evaluator | LLM-as-judge on TRACE level |
| Tool selection | `{prefix}_tool_selection` evaluator | LLM-as-judge on TOOL_CALL level |

**Next steps:**
- Deploy runtimes (`14-deploy-runtime.ipynb`), then re-enable eval: `npx cdk deploy OrgIQ-AgentCoreTelemetry --context enableEval=true`
- Adjust sampling rate in `agentcore-telemetry-stack.ts` → `samplingPercentage`
- Add per-agent filters in the `rule.filters` field to segment by service name

**Dashboard:** `orgiq-agentcore-telemetry` in CloudWatch Dashboards